# ACIS Framework — Quickstart Notebook
**Adversarial Construction Intelligence Security**


In [ ]:
from acis import ACISFramework, SystemProfile, AssetCategory
fw = ACISFramework()
print(fw)

## 1. Threat Assessment

In [ ]:
profile = SystemProfile(
    name='PPE Safety Monitor',
    asset_category=AssetCategory.SPS,
    uses_federated_learning=True,
    has_physical_consequence=False,
)
result = fw.assess_system(profile)
fw.print_report(result)

## 2. Risk Matrix

In [ ]:
from acis import ACISRiskMatrix
import pandas as pd
rm = ACISRiskMatrix()
df = rm.to_dataframe()
df.style.background_gradient(cmap='RdYlGn_r', vmin=1, vmax=5)

## 3. Data Poisoning Attack

In [ ]:
from acis.data import ConstructionBenchmark
from acis.attacks import ConstructionPPEPoison
from sklearn.ensemble import RandomForestClassifier

bench = ConstructionBenchmark()
ds    = bench.load_ppe(n_samples=1200)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(ds.X_train, ds.y_train)

attack = ConstructionPPEPoison(poison_rate=0.30)
result = attack.run(model, ds.as_tuple())
print(result.summary())

## 4. Model Extraction Attack

In [ ]:
from acis.attacks import ModelExtractionAttack
result = ModelExtractionAttack(n_queries=500).run(model, ds.as_tuple())
print(result.summary())
print(f'Fidelity: {result.metadata["fidelity"]:.1%}')

## 5. Federated Learning Security

In [ ]:
from acis.federated import FederatedCoordinator
coord   = FederatedCoordinator(n_rounds=10)
clients = coord.create_consortium(n_firms=8, n_malicious=2)
hist    = coord.train(clients, ds.X_train, ds.y_train, ds.X_test, ds.y_test)
coord.print_security_report(hist)